# PII Redaction (shadow -> enforce)

Sending raw user text to a third-party model can leak personal data. This
notebook shows a reversible redaction pipeline with two safety modes: `shadow`
(detect only, observe in production) and `enforce` (mask before sending,
restore on the way back).

## Tier-1: locale-neutral structured PII

The default detector is regex-based and catches structured PII -- email and
credit card (Luhn-checked). It deliberately does not catch free-form entities
like names; those need the optional Tier-2 NER extra. Keeping names out of the
default keeps the kit locale-neutral and dependency-free.

In [ ]:
import sys
sys.path.insert(0, "../src")

from genai_prod_kit.pii.detector import RegexDetector

text = "Reach me at taro@example.com or pay with card 4111 1111 1111 1111"
matches = RegexDetector().detect(text)
for m in matches:
    print(f"{m.pii_type.value:6s} conf={m.confidence}  {m.original_value!r}")

## shadow: detect, but send the original

In shadow mode the pipeline records what it would have masked but sends the
text unchanged. This lets you measure detection coverage in production before
you flip the switch that changes behaviour.

In [ ]:
from genai_prod_kit.pii.pipeline import run_pre_llm

shadow = run_pre_llm(text, mode="shadow")
print("matches      :", shadow.match_count)
print("sent (shadow):", shadow.redacted_text)

## enforce: mask out, then restore

In enforce mode the outgoing text is masked. After the model responds,
run_post_llm restores the placeholders and flags any that leaked through
unrestored.

In [ ]:
from genai_prod_kit.pii.pipeline import run_post_llm

enforce = run_pre_llm(text, mode="enforce")
print("sent (enforce):", enforce.redacted_text)

model_response = f"Sure, I will contact {enforce.redacted_text}"
restored, leaks = run_post_llm(model_response, enforce.redaction_map, mode="enforce")
print("restored      :", restored)
print("leaks         :", leaks)

## Rollout pattern

Ship in shadow first, watch the detection counts in your telemetry, confirm no
false positives matter, then promote to enforce. The reversible map means the
user still sees their real data in the final answer -- the model just never
saw it.